### Importação das bibliotecas

In [14]:
import pandas as pd

### Carregamento dos dados

In [15]:
df_vendas   = pd.read_csv('../data/processed/vendas_2023_2024_processed.csv')
df_produtos = pd.read_csv('../data/processed/produtos_processed.csv')

In [16]:
df_vendas.head(5)

,id_client,id_product,qtd,total,sale_date,cambio
0,42,105,11,3405.0,2023-09-10,4.9835
1,3,136,9,16873.9,2024-09-15,5.5717
2,25,139,7,9475.3,2024-08-13,5.4875
3,20,23,5,55893.0,2023-02-03,5.1030
4,8,57,4,451403.9,2024-02-12,4.9717


In [17]:
df_produtos.head(5)

,name,price,code,actual_category
0,Transponder AIS Maré Magnum,33122.52,1,eletrônicos
1,Transponder Furuno Marlin,13998.15,2,eletrônicos
2,Radar Furuno Pulse Leviathan,9024.19,3,eletrônicos
3,Rádio AIS Hydro Tidal Zen,3381.88,4,eletrônicos
4,Piloto Automático Furuno Storm,23669.01,5,eletrônicos


join nas tabelas para ter as vendas e as categorias

In [18]:
df = df_vendas.merge(
    df_produtos[['code', 'actual_category']],
    left_on='id_product',
    right_on='code',
    how='left'
).drop(columns='code')

df.head()

,id_client,id_product,qtd,total,sale_date,cambio,actual_category
0,42,105,11,3405.0,2023-09-10,4.9835,ancoragem
1,3,136,9,16873.9,2024-09-15,5.5717,ancoragem
2,25,139,7,9475.3,2024-08-13,5.4875,ancoragem
3,20,23,5,55893.0,2023-02-03,5.1030,eletrônicos
4,8,57,4,451403.9,2024-02-12,4.9717,propulsão


agrupar por clientes para obter as informaçoes solicitadas como faturamento total, frequencia e diversidade de categoria

In [21]:
df_clientes = df.groupby('id_client').agg(
  faturamento_total = ('total', 'sum'),
  frequencia = ('id_client', 'count'),
  diversidade_categorias = ('actual_category', 'nunique')
).reset_index()

df_clientes['ticket_medio'] = df_clientes['faturamento_total'] / df_clientes['frequencia']

df_clientes.head()

,id_client,faturamento_total,frequencia,diversidade_categorias,ticket_medio
0,1,51092500.05,190,3,268907.895000
1,2,65652931.35,220,3,298422.415227
2,3,59575349.10,207,3,287803.618841
3,4,50691754.40,207,3,244887.702415
4,5,58592802.70,202,3,290063.379703


selecionar a elite, apenas os melhores clientes de acordo com o que foi passado

In [23]:
df_elite = df_clientes[df_clientes['diversidade_categorias'] >= 3]

top10 = df_elite.sort_values(
    ['ticket_medio', 'id_client'],
    ascending=[False, True]
).head(10).reset_index(drop=True)

top10

,id_client,faturamento_total,frequencia,diversidade_categorias,ticket_medio
0,47,64003343.75,190,3,336859.703947
1,42,72187369.50,222,3,325168.331081
2,9,66788855.35,218,3,306370.896101
3,22,59581398.75,198,3,300916.155303
4,2,65652931.35,220,3,298422.415227
5,28,60826837.25,204,3,298170.770833
6,46,59126834.35,199,3,297119.770603
7,38,57093331.15,195,3,292786.313590
8,36,62791038.15,215,3,292051.340233
9,5,58592802.70,202,3,290063.379703


verificar a categoria dominante dos clientes

In [27]:
ids_client_top10 = top10['id_client'].tolist()

df_top10_vendas = df[df['id_client'].isin(ids_client_top10)]

categoria_dominante = df_top10_vendas.groupby('actual_category')['qtd'].sum().sort_values(ascending=False)

print("Quantidade de itens comprados de cada categoria (categoria dominante dos top10):")
print(categoria_dominante)

Quantidade de itens comprados de cada categoria (categoria dominante dos top10):
actual_category
propulsão      6030
ancoragem      5632
eletrônicos    5214
Name: qtd, dtype: int64
